# Multi-GRB lightcurve download + image pipeline

This notebook downloads Fermi GBM lightcurves for many GRBs and saves them as PNG/JPEG images so you can visually screen for FRED (Fast Rise Exponential Decay) profiles.

What it produces per GRB:
- one image per detector
- one combined multi-detector image

What it produces for the whole sample:
- a single summary grid image with the brightest detector per GRB
- a CSV of FRED scores (optional)

Run this notebook inside the Docker container (Jupyter Lab kernel: GRIPS Python 3.12).

In [ ]:
import sys
import os
import importlib
import pandas as pd

# Ensure sibling modules in this folder are importable
sys.path.insert(0, '/workspace/FRED_Classification')

from grb_discovery import query_gbm_catalog, build_grbs_df
import download_lc_pipeline as dlp
importlib.reload(dlp)

## 1. Choose / discover GRBs

Option A: query the Fermi GBM burst catalog for a date range.
Option B: load your own CSV with columns: `name, ra, dec, utc, sel_dets, t1, t2`.

In [ ]:
# Option A: discover from catalog
raw = query_gbm_catalog('2019-01-01', '2019-01-31')
grbs_df = build_grbs_df(raw)
print(f'{len(grbs_df)} GRBs found')
grbs_df.head()

In [ ]:
# Option B: load your own CSV (uncomment if needed)
# grbs_df = pd.read_csv('/workspace/data/my_grbs.csv')
# grbs_df['sel_dets'] = grbs_df['sel_dets'].apply(eval)  # string list -> real list
# print(f'{len(grbs_df)} GRBs loaded')

## 2. Download data and generate images

Adjust `output_dir`, `fmt`, `binsize`, `pre`, `post` as needed.

In [ ]:
output_dir = '/workspace/data/lightcurve_images'
fmt = 'png'              # or 'jpeg'
binsize = 0.064          # lightcurve bin size in seconds
pre = -20                # seconds before trigger
post = 50                # seconds after trigger

results, summary_path = dlp.run_pipeline(
    grbs_df,
    output_dir=output_dir,
    fmt=fmt,
    binsize=binsize,
    pre=pre,
    post=post,
    combined=True,       # save multi-detector combined plot per GRB
    summary=True,        # save single summary grid image
    score=True,          # print FRED scores per detector
)

## 3. Inspect results

Collect all per-detector FRED scores into a DataFrame and sort by score.

In [ ]:
rows = []
for r in results:
    rows.extend(r.get('scores', []))

scores_df = pd.DataFrame(rows)
if not scores_df.empty:
    scores_df = scores_df.sort_values('fred_score', ascending=False).reset_index(drop=True)
    display(scores_df[['grb_name', 'det', 'fred_score', 'rise_score', 'decay_score', 'single_score', 'n_prominent_peaks']].head(20))
else:
    print('No FRED scores available.')

In [ ]:
# Save scores to CSV for later filtering
if not scores_df.empty:
    scores_path = os.path.join(output_dir, 'fred_scores.csv')
    scores_df.to_csv(scores_path, index=False)
    print(f'Saved: {scores_path}')

## 4. Display the summary grid

Show the generated grid image in the notebook.

In [ ]:
from IPython.display import Image, display
if summary_path and os.path.exists(summary_path):
    display(Image(filename=summary_path))
else:
    print('No summary grid generated.')

## 5. Optional: process just one GRB

Use this if you want to re-run or inspect a single GRB without repeating the full batch.

In [ ]:
# Pick one row from the dataframe
row = grbs_df.iloc[0]
single_result = dlp.process_grb(
    row,
    output_dir=output_dir,
    fmt=fmt,
    binsize=binsize,
    pre=pre,
    post=post,
    combined=True,
    score=True,
)
single_result